# S2 STAR — institutional tearsheet

Frozen Universe D book from `s2_star_stack.json`. No further search.

**Focus:** after a single full-sample equity glance, almost everything below is **OOS** (dates after `RESEARCH_IS_END_STAR`). IS is only for the continuous curve kink, decay ratios, and IS-fit vol tercile cutpoints.

**Contract:** signal at close `t` → fill both legs at open `t+1`. Costs: `US_ALPACA_D_REALISTIC`. Rolling IC is not used — reversion health is ADF / half-life / IS→OOS Sharpe decay.

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IPython.display import display

from backtest.s2_coint.diagnosis import (
    check_fill_timing,
    enrich_trades,
    extreme_trades,
    gross_returns_from_net,
    plotly_pair_diagnosis,
    print_extreme_trades,
)
from backtest.s2_coint.report import load_star_stack
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_star_panels,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_ledger_total,
    overlay_ols_hedge,
    repo_root,
    split_is_oos,
    star_panel_paths,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.tearsheet import (
    EVENT_WINDOWS,
    beta_corr_to_benchmark,
    bootstrap_sharpe_ci,
    capacity_table,
    cost_stress_oos_sharpes,
    cvar,
    daily_regime_performance,
    dividend_split_audit,
    equity_from_returns,
    export_s2_period_returns,
    fit_mean_abs_score,
    gap_vs_path_decomposition,
    max_consecutive_losers,
    monthly_returns,
    oos_headline_metrics,
    pair_pnl_attribution,
    pair_scorecard,
    plot_drawdown,
    plot_equity_net_gross,
    plot_event_grid,
    plot_gap_bucket_means,
    plot_losing_months,
    plot_monthly_heatmap,
    plot_static_weekly_overlay,
    plotly_weekly_overlay,
    reversion_health_table,
    split_returns_at_is_end,
    spy_daily_returns,
    utilization_stats,
    weekly_overlay_frame,
    write_star_tearsheet_pdf,
)
from performance.metrics import plot_rolling_metrics, rolling_sharpe_ratio
from strategies.s2_coint.metrics import metrics_from_returns

ROOT = repo_root(ROOT)
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
UNIVERSE = str(stack["UNIVERSE_STAR"])
BAR = str(stack.get("BAR_STAR") or "1d")
PAIRS = list(stack.get("PAIRS_STAR") or [])
N_TRIALS_LOCAL = 1
N_TRIALS_STACK = n_trials_ledger_total()
PDF_PATH = os.path.join(ARTIFACTS_DIR, "s2_star_tearsheet.pdf")
S2_RETURNS_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s2_coint", "s2_period_returns.parquet"
)

print(f"ROOT={ROOT}")
print(f"STAR={STAR_PATH}")
print(f"n_trials_local={N_TRIALS_LOCAL}  n_trials_stack={N_TRIALS_STACK}")
display(pd.Series(stack, name="value").to_frame())


## 1. Contract

In [ ]:
print("Timing: close-t signal → open-(t+1) fill (both legs)")
print(f"Cost profile: {stack.get('COST_PROFILE_STAR') or 'US_ALPACA_D_REALISTIC (Universe D default)'}")
print("Not modelled: HTB locate spikes, vol-scaled slippage, single-leg dividends in PnL, overnight auction microstructure")
print(f"Pairs: {PAIRS}")
print(f"Research IS end: {stack.get('RESEARCH_IS_END_STAR')}")


## 2. Run

`mean_abs_score` is the in-sample mean of `|z|` used to scale score-based position size so a typical signal is order-1; it is frozen on IS and not refit on OOS.

In [ ]:
# Load STAR panels when cached; else rebuild OLS overlay from census panel.
try:
    train, full, manifest = load_star_panels(
        universe=UNIVERSE, bar=BAR, pair_ids=PAIRS, root=ROOT
    )
    print("loaded star panels", manifest.get("stack_hash"))
except (FileNotFoundError, ValueError) as exc:
    warnings.warn(f"star panels unavailable ({exc}); overlaying OLS from census panel")
    train, full = load_universe_panels(UNIVERSE, BAR, PAIRS, root=ROOT)
    lb = lookbacks_for_bar(
        BAR,
        ols_days=int(stack.get("OLS_WINDOW_STAR") or 252),
        z_days=int(stack.get("Z_WINDOW_STAR") or 90),
        adf_days=int(stack.get("ADF_WINDOW_STAR") or 252),
    )
    train = overlay_ols_hedge(
        train,
        ols_window=lb["ols_window"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )
    full = overlay_ols_hedge(
        full,
        ols_window=lb["ols_window"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )

IS_END = is_end_for_stack(stack, full)
is_panel, oos_panel = split_is_oos(full, is_end=IS_END)
cfg = config_from_stack(stack)
mean_abs = fit_mean_abs_score(is_panel, score_column=cfg.score_column)
s1_weekly = load_s1_weekly(ROOT)
spy = spy_daily_returns(full["date"].min(), full["date"].max())

print(f"IS_END={IS_END.date()}  mean_abs_score={mean_abs:.4f}")
print(f"full rows={len(full)}  IS={len(is_panel)}  OOS={len(oos_panel)}")
print(f"cost_profile={cfg.cost_profile}  size_mode={cfg.size_mode}  vol_mode={cfg.vol_mode}")

full_res = run_s2_backtest(
    full,
    cfg,
    s1_weekly=s1_weekly,
    mean_abs_score=mean_abs,
    n_trials_local=N_TRIALS_LOCAL,
    n_trials_stack=N_TRIALS_STACK,
)
oos_res = run_s2_backtest(
    oos_panel,
    cfg,
    s1_weekly=s1_weekly,
    mean_abs_score=mean_abs,
    n_trials_local=N_TRIALS_LOCAL,
    n_trials_stack=N_TRIALS_STACK,
)

full_net = full_res.returns
oos_net = oos_res.returns
oos_gross = gross_returns_from_net(oos_net, oos_res.pair_trades)
is_net, _ = split_returns_at_is_end(full_net, IS_END)

# Enrich OOS trades with signal context
oos_trades_enriched_parts = []
for pid, pr in oos_res.book.pair_results.items():
    g = oos_panel.loc[oos_panel["pair_id"].astype(str) == str(pid)]
    enr = enrich_trades(pr.trades, g, pr.returns)
    if not enr.empty:
        oos_trades_enriched_parts.append(enr)
oos_trades = (
    pd.concat(oos_trades_enriched_parts, ignore_index=True)
    if oos_trades_enriched_parts
    else pd.DataFrame()
)

print("full metrics", full_res.metrics)
print("oos metrics", oos_res.metrics)
print(f"OOS trades={len(oos_trades)}")


## 3. Full-sample glance (then stop using it)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
eq = equity_from_returns(full_net)
ax.plot(eq.index, eq.values, color="#1f4e79", lw=1.8, label="net")
ax.axvline(IS_END, color="#333333", ls="--", lw=1.0, zorder=5, label="IS end")
ax.set_title("Full-sample net equity")
ax.set_ylabel("Equity")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
is_eq = equity_from_returns(is_net)
oos_eq = equity_from_returns(oos_net)
if not is_eq.empty:
    ax.plot(is_eq.index, is_eq.values, color="#5b7c99", lw=1.5, label="IS")
if not oos_eq.empty:
    ax.plot(oos_eq.index, oos_eq.values, color="#1f4e79", lw=1.8, label="OOS")
ax.axvline(IS_END, color="#333333", ls="--", lw=1.0, zorder=5)
ax.set_title("IS and OOS each rebased to 1.0")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.show()

is_m = metrics_from_returns(is_net)
oos_m = metrics_from_returns(oos_net)
decay = pd.Series(
    {
        "IS_sharpe": is_m["ann_sharpe"],
        "OOS_sharpe": oos_m["ann_sharpe"],
        "OOS_over_IS": (
            oos_m["ann_sharpe"] / is_m["ann_sharpe"]
            if np.isfinite(is_m["ann_sharpe"]) and abs(is_m["ann_sharpe"]) > 1e-12
            else float("nan")
        ),
    }
)
display(decay.to_frame("value"))


## 4. OOS headline

In [ ]:
headline = oos_headline_metrics(
    oos_net,
    oos_gross,
    trades=oos_res.pair_trades,
    s1_weekly=s1_weekly,
    spy_daily=spy,
    n_trials_local=N_TRIALS_LOCAL,
    n_trials_stack=N_TRIALS_STACK,
)
display(headline.to_frame("OOS"))


## 5. OOS standalone performance

### 5.1 OOS equity (net vs gross)

In [ ]:
fig = plot_equity_net_gross(oos_net, oos_gross, title="OOS equity — net vs gross")
plt.show()


### 5.2 Drawdown

In [ ]:
fig = plot_drawdown(oos_net, title="OOS drawdown (net)")
plt.show()


### 5.3 Rolling Sharpe

In [ ]:
fig = plot_rolling_metrics(oos_net, metrics=("sharpe",), window=63)
plt.show()


### 5.4 Monthly heatmap

In [ ]:
oos_monthly = monthly_returns(oos_net)
fig = plot_monthly_heatmap(oos_monthly, title="OOS monthly heatmap")
plt.show()


### 5.5 Return distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(oos_net.dropna(), bins=40, color="#1f4e79", alpha=0.85)
ax.set_title("OOS daily net returns")
ax.grid(True, alpha=0.3)
plt.show()
worst_days = oos_net.nsmallest(10).to_frame("ret")
display(worst_days)


## 6. Overlay (secondary)

In [ ]:
weekly_full = weekly_overlay_frame(full_net, s1_weekly, spy)
weekly_oos = weekly_overlay_frame(oos_net, s1_weekly, spy)
print(f"weekly_full n={len(weekly_full)}  weekly_oos n={len(weekly_oos)}")

try:
    fig = plotly_weekly_overlay(weekly_full, weekly_oos, title="S2 vs S1 vs SPY (weekly)")
    fig.show()
except Exception as exc:
    warnings.warn(f"plotly overlay failed ({exc}); static fallback")
    fig = plot_static_weekly_overlay(weekly_oos, title="S2 vs S1 vs SPY — OOS weekly", is_end=IS_END)
    plt.show()


In [ ]:
# Excess S2 − SPY (daily OOS) and beta/corr readout
spy_oos = spy.reindex(oos_net.index).fillna(0.0)
excess = (oos_net.fillna(0.0) - spy_oos).rename("excess")
fig = plot_equity_net_gross(excess, excess, title="OOS excess equity (S2 − SPY)")
plt.show()
display(pd.Series(beta_corr_to_benchmark(oos_net, spy), name="value").to_frame())


## 7. OOS losing months

In [ ]:
fig = plot_losing_months(oos_monthly, title="OOS monthly returns")
plt.show()
display(oos_monthly.sort_values().head(10).to_frame("monthly_return"))


## 8. OOS regimes

In [ ]:
is_dates = pd.DatetimeIndex(pd.to_datetime(is_panel["date"])).unique()
regimes = daily_regime_performance(oos_net, spy, is_dates)
print("Vol cutpoints (IS):")
display(regimes["vol_cutpoints"].to_frame("value"))
print("By market direction:")
display(regimes["market"])
print("By vol tercile:")
display(regimes["vol"])


## 9. Event study

In [ ]:
fig = plot_event_grid(full_net, is_end=IS_END, windows=EVENT_WINDOWS)
plt.show()


## 10. Reversion health (no IC)

In [ ]:
health = reversion_health_table(full, is_end=IS_END)
display(health)

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
for pid, g in full.groupby("pair_id", sort=False):
    g = g.sort_values("date")
    axes[0].plot(g["date"], g["adf_pvalue"], lw=1.0, label=str(pid))
    axes[1].plot(g["date"], g["half_life"], lw=1.0, label=str(pid))
axes[0].axhline(0.05, color="gray", ls=":", lw=0.8)
axes[0].axvline(IS_END, color="#333333", ls="--", lw=1.0)
axes[1].axvline(IS_END, color="#333333", ls="--", lw=1.0)
axes[0].set_title("Rolling ADF p-value")
axes[1].set_title("Half-life (bars)")
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

print(
    f"IS Sharpe={is_m['ann_sharpe']:.3f} → OOS Sharpe={oos_m['ann_sharpe']:.3f} "
    f"(ratio={decay['OOS_over_IS']:.3f})"
)


## 11. OOS pair scorecard

In [ ]:
scorecard = pair_scorecard(oos_res.book.pair_results, full, is_end=IS_END)
display(scorecard)
attr = pair_pnl_attribution(oos_res.book.pair_results, is_end=IS_END)
display(attr.to_frame("oos_total_return"))
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(attr.index), 100.0 * attr.values, color="#1f4e79")
ax.set_title("OOS pair attribution (% total return)")
ax.grid(True, axis="x", alpha=0.3)
plt.show()


## 12. Trades — losing first

In [ ]:
N_EXTREME_TRADES = 7  # edit this constant to show more/fewer extreme trades

worst, best = extreme_trades(oos_trades, n=N_EXTREME_TRADES)
print(f"=== Worst {N_EXTREME_TRADES} OOS trades ===")
display(worst)
print(f"=== Best {N_EXTREME_TRADES} OOS trades ===")
display(best)

# Drivers for large |pnl| trades (~20%+)
big = oos_trades.loc[oos_trades["pnl_pct"].abs() >= 20.0].sort_values("pnl_pct")
if not big.empty:
    print("Large |pnl| >= 20% — drivers")
    cols = [
        c
        for c in [
            "pair_id",
            "side_label",
            "entry_date",
            "exit_date",
            "pnl_pct",
            "exit_reason",
            "z_entry",
            "z_exit",
            "adf_entry",
            "adf_exit",
            "hold_bars",
        ]
        if c in big.columns
    ]
    display(big[cols])
else:
    print("No OOS trades with |pnl_pct| >= 20")


## 13. Interactive spreads

In [ ]:
for pid in PAIRS:
    g = full.loc[full["pair_id"].astype(str) == str(pid)].sort_values("date")
    t = oos_trades.loc[oos_trades["pair_id"].astype(str) == str(pid)] if not oos_trades.empty else oos_trades
    # Include full-sample trades for markers across IS/OOS on the chart
    all_parts = []
    pr = full_res.book.pair_results.get(str(pid))
    if pr is not None and not pr.trades.empty:
        all_parts.append(enrich_trades(pr.trades, g, pr.returns))
    trades_plot = pd.concat(all_parts, ignore_index=True) if all_parts else t
    pair_ret = pr.returns if pr is not None else None
    fig = plotly_pair_diagnosis(
        g,
        trades_plot,
        entry_z=float(cfg.entry_z),
        z_window=int(cfg.z_window),
        pair_returns=pair_ret,
        is_end=IS_END,
        title=f"{pid} STAR diagnosis",
    )
    fig.show()


## 14. Friction

In [ ]:
util = utilization_stats(oos_net, oos_res.pair_trades, n_slots=6)
friction = pd.Series(
    {
        "ann_sharpe_gross": headline["ann_sharpe_gross"],
        "ann_sharpe_net": headline["ann_sharpe_net"],
        "cost_bps_year": headline["cost_bps_year"],
        **util.to_dict(),
    }
)
display(friction.to_frame("value"))

stress = cost_stress_oos_sharpes(
    oos_panel,
    cfg,
    mean_abs_score=mean_abs,
    s1_weekly=s1_weekly,
    n_trials_local=N_TRIALS_LOCAL,
    n_trials_stack=N_TRIALS_STACK,
)
display(stress)


## 15. Capacity

In [ ]:
tickers = sorted({leg for pid in PAIRS for leg in str(pid).split("|")})
cap = capacity_table(
    tickers,
    start=oos_panel["date"].min(),
    end=oos_panel["date"].max(),
    book_notional=1_000_000.0,
    n_slots=6,
)
display(cap)


## 16. Tail + Sharpe CI

**Tail:** the worst left-hand outcomes (largest losing days, consecutive losers), not the average day.

**Sharpe CI:** the printed OOS Sharpe is one draw from a short sample. Bootstrap resamples those daily net returns with replacement, recomputes Sharpe each time, and reports a 95% band. If the band includes 0 or 1.0, do not treat the point Sharpe as a sure edge. PSR (`P(true SR > 1)`) is a different question (hurdle probability, not a range around the point estimate).

**CVaR (expected shortfall):** the mean of returns in the worst α of days (here 5%). VaR is the cutoff of that tail; CVaR is the average loss *beyond* that cutoff — how bad the bad days are, not just how often they happen.

In [ ]:
cvar5 = cvar(oos_net, alpha=0.05)
consec = max_consecutive_losers(oos_net)
ci = bootstrap_sharpe_ci(oos_net, n_boot=2000, seed=42)
tail = pd.Series(
    {
        "cvar_5": cvar5,
        "max_consecutive_losers": consec,
        "sharpe": ci["sharpe"],
        "sharpe_ci_low": ci["ci_low"],
        "sharpe_ci_high": ci["ci_high"],
        "psr": headline["psr"],
        "dsr_stack": headline["dsr_stack"],
    }
)
display(tail.to_frame("value"))
print("Worst trades: see section 12")


## 17. Gap vs session path — weekend vs weekday overnight

Engine PnL is open-to-open. Open days are split into **session** (`open→close`), **weekday overnight** (`close→next open` with no weekend), and **weekend** (gap that spans Saturday/Sunday).

In [ ]:
gap = gap_vs_path_decomposition(
    full,
    oos_trades if not oos_trades.empty else oos_res.pair_trades,
    is_end=IS_END,
    use_hedge_ratio_sizing=cfg.use_hedge_ratio_sizing,
)
display(gap)
fig = plot_gap_bucket_means(gap, title="OOS mean return by gap bucket (weekend vs weekday overnight)")
plt.show()


## 18. Dividend / split audit

In [ ]:
audit = dividend_split_audit(full, oos_trades if not oos_trades.empty else oos_res.pair_trades)
flagged = audit.loc[audit["n_jump_days"] > 0].sort_values("max_abs_jump", ascending=False)
print(f"trades overlapping large raw jumps: {len(flagged)} / {len(audit)}")
display(flagged.head(20) if not flagged.empty else audit.head())


## 19. Fill-timing check

In [ ]:
timing = check_fill_timing(
    oos_res.pair_trades if not oos_res.pair_trades.empty else oos_trades,
    oos_panel,
)
display(timing.head())
n_bad = int((~timing["all_ok"]).sum()) if not timing.empty else 0
print(f"fill-timing failures: {n_bad} / {len(timing)}")
if n_bad:
    raise AssertionError(f"{n_bad} trades failed close-t → open-t+1 fill-timing checks")


## 20. Export + PDF

In [ ]:
path = export_s2_period_returns(oos_net, S2_RETURNS_PATH)
print(f"wrote daily S2 net returns -> {path}  n={len(oos_net)}")

write_star_tearsheet_pdf(
    PDF_PATH,
    stack=stack,
    is_end=IS_END,
    full_net=full_net,
    oos_net=oos_net,
    oos_gross=oos_gross,
    headline=headline,
    weekly_full=weekly_full,
    weekly_oos=weekly_oos,
    spy_daily=spy,
    event_returns=full_net,
    health=health,
    attribution=attr,
    worst_trades=worst,
    best_trades=best,
    capacity=cap,
    gap_table=gap,
    sharpe_ci=ci,
    cvar_5=cvar5,
    title="S2 STAR — institutional tearsheet (Universe D)",
)
print(f"wrote desk PDF -> {PDF_PATH}")
